# 🗺️ Car-Sharing & Urban Mobility — Lab 2

**Politecnico di Torino — ICT for Smart Mobility**  
**Group Project — Individual Contribution**

This notebook presents the analytical work independently developed by **Sima Shahbazian** from **Lab 2, Part 2 through the end of the lab**.

### 🔎 Main analyses
- 🚗 Car-sharing rental validation
- 🗺️ Origin-Destination (OD) matrices for Turin
- 📏 L2 normalization
- 🔥 OD-matrix visualization
- 📐 Four similarity metrics
- 👥 Gender, age and travel-motivation segmentation
- 🚘 IMQ / UnipolTech / Car2Go / Enjoy comparison

> 🔐 Private MongoDB credentials, university connection details and local machine paths are removed from this public portfolio version. The result tables below reproduce the results of the submitted analysis.


## 🧰 Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from numpy.linalg import norm


## 🚗 1. Car-Sharing Rental Validation

The original Lab 2 analysis validated the Turin Car2Go and Enjoy rental records by checking:
- trips shorter than 2 minutes
- trips longer than 24 hours
- whether origin and destination represented actual vehicle movement

The analysis reported no short or long rental-duration anomalies in the examined datasets.


In [ ]:
def validate_rentals(data, duration_col):
    duration = pd.to_numeric(data[duration_col], errors="coerce")
    return pd.Series({
        "Total records": len(data),
        "Under 2 min": int((duration < 120).sum()),
        "Over 24 h": int((duration > 86400).sum())
    })

def filter_moving_trips(data, origin_col, destination_col):
    return data[data[origin_col] != data[destination_col]].copy()

# Example with locally prepared data:
# validation = validate_rentals(bookings, "duration_seconds")
# display(validation.to_frame("Value"))


## 🗺️ 2. Origin-Destination Matrices

The analysis used **23 predefined Turin zones** from `TorinoZonesArray.geojson`.

For each dataset, trips were aggregated by origin and destination zone and the resulting OD matrix was L2-normalized before comparison.


In [ ]:
def build_od_matrix(data, origin_col, destination_col):
    return (
        data.groupby([origin_col, destination_col])
        .size()
        .unstack(fill_value=0)
    )

def normalize_matrix(matrix):
    values = np.asarray(matrix, dtype=float)
    l2 = np.linalg.norm(values)
    if l2 == 0:
        raise ValueError("Cannot normalize an all-zero matrix.")
    return values / l2

# Example:
# od = build_od_matrix(data, "ORI_ZONE", "DST_ZONE")
# od_normalized = normalize_matrix(od)


### 📊 Overall IMQ vs. UnipolTech

The original analysis produced normalized OD matrices for both datasets and compared them using Euclidean distance.


In [ ]:
overall_imq_unipol = pd.DataFrame({
    "Comparison": ["IMQ vs UnipolTech"],
    "Euclidean Distance": [0.43709507883479234]
})
display(overall_imq_unipol)


## 🔥 3. OD Matrix Visualization

Heatmaps were used to inspect spatial mobility patterns and traffic concentration across Turin zones.

The same visualization method can be applied to the normalized matrices generated above.


In [ ]:
def plot_od_heatmap(matrix, title):
    plt.figure(figsize=(9, 7))
    plt.imshow(matrix, aspect="auto")
    plt.colorbar(label="Normalized trip intensity")
    plt.title(title)
    plt.xlabel("Destination zone")
    plt.ylabel("Origin zone")
    plt.tight_layout()
    plt.show()

# Example:
# plot_od_heatmap(od_normalized, "Normalized OD Matrix")


## 🕐 4. Temporal OD Comparisons

The OD matrices were compared for:
- 📅 Weekdays vs. weekends
- 📆 First week vs. second week (weekdays)
- 🚘 Car2Go vs. Enjoy
- 🎲 Random matrices as a dissimilarity reference


### 📐 Similarity Metrics

In [ ]:
def sad(matrix1, matrix2):
    return np.sum(np.abs(matrix1 - matrix2))

def euclidean_distance(matrix1, matrix2):
    return np.sqrt(np.sum((matrix1 - matrix2) ** 2))

def maximum_norm(matrix1, matrix2):
    return np.max(np.abs(matrix1 - matrix2))

def spectral_norm(matrix1, matrix2):
    diff = matrix1 - matrix2
    return np.max(norm(diff, axis=1))

def calculate_similarity(matrix1, matrix2):
    return {
        "SAD": sad(matrix1, matrix2),
        "Euclidean": euclidean_distance(matrix1, matrix2),
        "Maximum Norm": maximum_norm(matrix1, matrix2),
        "Spectral Norm": spectral_norm(matrix1, matrix2)
    }


### 📊 Reported similarity results

In [ ]:
similarity_results = pd.DataFrame([
    ["Weekdays vs Weekends", 2.5065656252508817, 0.20142432426847924, 0.09966423335352909, 0.1285474760796762],
    ["First Week vs Second Week", 2.5121313648430144, 0.16284752878922925, 0.03757442375868688, 0.0560842835881698],
    ["Car2Go vs Enjoy", 4.594291012056778, 0.3494539303101945, 0.17641969540665408, 0.22232848949291474],
    ["Random Matrices", 181.07451195789403, 9.511474268564365, 0.9628805579072461, 2.669796839081532]
], columns=["Comparison","SAD","Euclidean","Maximum Norm","Spectral Norm"])

display(similarity_results.round(4))


## 👥 5. Behavioral Segmentation

The Lab 2 analysis generated and compared OD matrices for:

- 👩 Gender
- 🎂 Age groups
- 💼 Work vs. non-work travel motivation

The matrices were normalized and compared using Euclidean distance.


### 👩 Gender — IMQ vs UnipolTech

In [ ]:
gender_results = pd.DataFrame([
    ["Female", 0.4505463228619883],
    ["Male", 0.5153452920550164]
], columns=["Gender", "Euclidean Distance"])

display(gender_results.round(4))


### 🎂 Age groups — IMQ vs UnipolTech

The groups used in the original analysis were:
- Group 1: 11–19
- Group 2: 20–49
- Group 3: 50–69
- Group 4: 70+


In [ ]:
age_results = pd.DataFrame([
    ["Group 1", "11–19", 0.9415304260853421],
    ["Group 2", "20–49", 0.5598477373645924],
    ["Group 3", "50–69", 0.48808003616945334],
    ["Group 4", "70+", 0.37706531882241834]
], columns=["Age Group", "Age Range", "Euclidean Distance"])

display(age_results.round(4))


### 💼 Travel motivation — IMQ vs UnipolTech

In [ ]:
motivation_results = pd.DataFrame([
    ["Work", 0.7954906105831622],
    ["Non-work", 0.4236097167728805]
], columns=["Motivation", "Euclidean Distance"])

display(motivation_results.round(4))


### 👩‍🦰👨 Gender differences across age groups — IMQ

The original analysis compared age Group 1 with Group 3 separately for female and male users.


In [ ]:
gender_age_results = pd.DataFrame([
    ["Female", "Group 1 vs Group 3", 0.7417183655618514],
    ["Male", "Group 1 vs Group 3", 0.6774232433244338]
], columns=["Gender", "Comparison", "Euclidean Distance"])

display(gender_age_results.round(4))


## 🚘 6. IMQ Users vs Car2Go and Enjoy

The original analysis compared the normalized IMQ OD matrix with the overall Car2Go and Enjoy matrices and then repeated the comparison for selected user segments.

### Overall


In [ ]:
overall_services = pd.DataFrame([
    ["IMQ", "Car2Go", 0.781984236858352],
    ["IMQ", "Enjoy", 0.799554403588304],
    ["Car2Go", "Enjoy", 0.327803600850015]
], columns=["Dataset A", "Dataset B", "Euclidean Distance"])

display(overall_services.round(4))


### 👩 Female users

In [ ]:
female_services = pd.DataFrame([
    ["Female IMQ", "Car2Go", 0.8252039294038613],
    ["Female IMQ", "Enjoy", 0.8452442104243314]
], columns=["IMQ Segment", "Service", "Euclidean Distance"])

display(female_services.round(4))


### 👨 Male users

In [ ]:
male_services = pd.DataFrame([
    ["Male IMQ", "Car2Go", 0.7454719749700488],
    ["Male IMQ", "Enjoy", 0.7587045561921734]
], columns=["IMQ Segment", "Service", "Euclidean Distance"])

display(male_services.round(4))


### 💼 Work-related motivation

In [ ]:
work_services = pd.DataFrame([
    ["Work IMQ", "Car2Go", 0.6691079266012425],
    ["Work IMQ", "Enjoy", 0.7015789757914659]
], columns=["IMQ Segment", "Service", "Euclidean Distance"])

display(work_services.round(4))


### 🛍️ Non-work motivation

In [ ]:
nonwork_services = pd.DataFrame([
    ["Non-work IMQ", "Car2Go", 0.8219469734604784],
    ["Non-work IMQ", "Enjoy", 0.8367138565437258]
], columns=["IMQ Segment", "Service", "Euclidean Distance"])

display(nonwork_services.round(4))


### 🎂 Age-group comparisons with Car2Go and Enjoy

In [ ]:
age_service_results = pd.DataFrame([
    ["Age Group 1", "Car2Go", 0.7322403223073854],
    ["Age Group 1", "Enjoy", 0.7563780027143918],
    ["Age Group 2", "Car2Go", 0.7991186005057944],
    ["Age Group 2", "Enjoy", 0.8001999331279194],
    ["Age Group 3", "Car2Go", 0.7969570818110624],
    ["Age Group 3", "Enjoy", 0.811226856155219],
], columns=["Age Group", "Service", "Euclidean Distance"])

display(age_service_results.round(4))


## 🌞 7. Daytime Comparison

The submitted analysis also separated day and night periods and compared the daytime IMQ mobility structure with Car2Go and Enjoy.


In [ ]:
daytime_services = pd.DataFrame([
    ["Daytime IMQ", "Car2Go", 0.8175949487980514],
    ["Daytime IMQ", "Enjoy", 0.8308379468517807]
], columns=["IMQ Segment", "Service", "Euclidean Distance"])

display(daytime_services.round(4))


## 📊 8. Overall Dataset Comparison

The original report also compared total normalized trip volumes:

| Dataset | Total trips represented |
|---|---:|
| UnipolTech | 9.11 |
| Car2Go | 11.92 |
| Enjoy | 13.73 |
| IMQ | 12.016 |

The report additionally reported Euclidean distances between the overall datasets:


In [ ]:
overall_dataset_results = pd.DataFrame([
    ["IMQ", "Car2Go", 0.781984236858352],
    ["IMQ", "Enjoy", 0.799554403588304],
    ["UnipolTech", "Car2Go", 1.0182201679895946],
    ["UnipolTech", "Enjoy", 1.013685862669674],
    ["Car2Go", "Enjoy", 0.327803600850015]
], columns=["Dataset A", "Dataset B", "Euclidean Distance"])

display(overall_dataset_results.round(4))


## 🧠 9. Interpretation

The numerical results provide several observations from the submitted analysis:

- The **weekday/weekend** and **first-week/second-week** OD comparisons produced relatively small distances.
- **Car2Go vs. Enjoy** had a larger distance than the two temporal comparisons.
- For **IMQ vs. UnipolTech**, the reported gender distances were 0.4505 for females and 0.5153 for males.
- Across the four age groups, the reported distance decreased from Group 1 to Group 4.
- Work-related motivation showed a larger IMQ–UnipolTech distance than non-work motivation.
- In the service comparisons shown above, the IMQ distances to **Car2Go were lower than to Enjoy** for the overall, female, male, work, non-work and daytime comparisons in the submitted results.

> These are descriptive comparisons from the course project. They do not establish causal explanations for the observed differences.


## 🔐 Portfolio / Data Note

The original notebook contains university database access, local paths and course datasets. Those elements are intentionally not included here.

The public version preserves the **analytical logic, methods, result tables and reported numerical outputs** while removing private access information.

**Individual contribution:** Lab 2, Part 2 through the end.
